In [1]:
# %%
#
# Cell 1: Initial Setup
#
import pandas as pd
import numpy as np
import torch
import random
import os

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorWithPadding
)

from datasets import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    matthews_corrcoef, balanced_accuracy_score,
    cohen_kappa_score, jaccard_score, hamming_loss,
    confusion_matrix
)



/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
2025-06-28 19:23:30.529998: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-28 19:23:31.367143: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
202

In [2]:
# %%
#
# Cell 2: W&B and Hugging Face Login
#
import wandb
import optuna
import huggingface_hub

import os
os.environ["WANDB_PROJECT"] = "distilbert_degendered"

# NOTE: Replace with your actual keys or use environment variables
# wandb.login(key="YOUR_WANDB_KEY")
# huggingface_hub.login(token="YOUR_HF_TOKEN")


In [3]:
# %%
#
# Cell 3: Model Configuration
#
model_name = "distilbert-base-uncased"
model_cache_path = "../scratch/cache/distilbert_degendered"



In [4]:
# %%
#
# Cell 4: Data Preparation
#
# Ensure 'data/combined_letters_degendered.csv' exists at the specified path
df = pd.read_csv("data/combined_letters_degendered.csv")[["full_text", "label"]].dropna()
df["label"] = df["label"].astype(int)

# Perform train-test split, stratifying by label to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(
    df["full_text"],
    df["label"],
    test_size=0.2,
    stratify=df["label"],
)

# The tokenizer is always loaded from the base model
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_cache_path)



tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [5]:
# %%
#
# Cell 5: Tokenization
#
# This function prepares your text data for the model by converting it into token IDs
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding=False,
        max_length=512
    )
    tokens["labels"] = example["label"]
    return tokens

# Convert pandas Series to Hugging Face Dataset objects, then map the tokenization function
train_dataset = Dataset.from_dict({"text": X_train.tolist(), "label": y_train.tolist()})
test_dataset = Dataset.from_dict({"text": X_test.tolist(), "label": y_test.tolist()})

tokenized_train = train_dataset.map(tokenize, batched=True).remove_columns(["text"])
tokenized_test = test_dataset.map(tokenize, batched=True).remove_columns(["text"])

# Data Collator: Dynamically pads input sequences to the longest sequence in the batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)



Map:   0%|          | 0/7189 [00:00<?, ? examples/s]

Map:   0%|          | 0/1798 [00:00<?, ? examples/s]

In [6]:
# %%
#
# Cell 6: Metrics Function
#
# Compute metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    preds = logits.argmax(-1)
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)

    print("Confusion Matrix:", confusion_matrix(labels, preds))

    return {
        "f1_score": f1,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "mcc": matthews_corrcoef(labels, preds),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "cohen_kappa": cohen_kappa_score(labels, preds),
        "jaccard": jaccard_score(labels, preds, average="macro"),
        "hamming_loss": hamming_loss(labels, preds)
    }



In [7]:
# %%
#
# Cell 7: Model Initialization for Hyperparameter Optimization
#
def model_init(trial=None):
    # Load the base pre-trained model (DistilBERT)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label={0: "female", 1: "male"},
        label2id={"female": 0, "male": 1},
        cache_dir=model_cache_path,
        device_map="auto"
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    return model



In [8]:
# %%
#
# Cell 8: Optuna Hyperparameter Space
#
def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 10),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32]),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.03, step=0.01),
    }



In [9]:
# %%
#
# Cell 9: Trainer for Hyperparameter Optimization
#
training_args_for_hpo = TrainingArguments(
    output_dir="../scratch/hpo_results_distilbert_degendered",
    per_device_eval_batch_size=32,
    fp16=True,
    save_strategy="no",
    logging_steps=50,
    report_to="wandb",
    remove_unused_columns=False,
    load_best_model_at_end=False,
    eval_strategy="epoch",
)

trainer = Trainer(
    model_init=model_init,
    args=training_args_for_hpo,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)



/tmp/ipykernel_2387882/809944248.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
# %%
#
# Cell 10: Run Hyperparameter Search
#
best_run = trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=optuna_hp_space,
    n_trials=20,
    compute_objective=lambda metrics: metrics["eval_f1_score"]
)

print("Best run details:")
print(best_run)

# Access the best hyperparameters found by Optuna
best_hps = best_run.hyperparameters
print("Best Hyperparameters Found:")
for hp, value in best_hps.items():
    print(f"  {hp}: {value}")

# W&B will provide a URL to the best run in its logs.
if hasattr(best_run, 'url'):
    print(f"Find the best run and explore all trials in W&B at: {best_run.url}")



[I 2025-06-28 19:24:02,650] A new study created in memory with name: no-name-f5edc67d-bf8f-408f-90e5-6566d3139f09
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: mtwesley to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,F1 Score,Accuracy,Precision,Recall,Mcc,Balanced Accuracy,Cohen Kappa,Jaccard,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.623700,0.619755,0.563707,0.690211,0.476392,0.690211,0.000000,0.500000,0.000000,0.345106,0.309789,1.058700,1698.327000,53.840000
2,0.624200,0.619262,0.563707,0.690211,0.476392,0.690211,0.000000,0.500000,0.000000,0.345106,0.309789,1.049500,1713.163000,54.311000
3,0.614900,0.620379,0.563707,0.690211,0.476392,0.690211,0.000000,0.500000,0.000000,0.345106,0.309789,1.067000,1685.021000,53.418000
4,0.601500,0.620214,0.563707,0.690211,0.476392,0.690211,0.000000,0.500000,0.000000,0.345106,0.309789,1.047000,1717.263000,54.440000
5,0.607700,0.618933,0.563707,0.690211,0.476392,0.690211,0.000000,0.500000,0.000000,0.345106,0.309789,1.048300,1715.237000,54.376000


Confusion Matrix: [[   0  557]
 [   0 1241]]
Confusion Matrix: [[   0  557]
 [   0 1241]]
Confusion Matrix: [[   0  557]
 [   0 1241]]
Confusion Matrix: [[   0  557]
 [   0 1241]]
Confusion Matrix: [[   0  557]
 [   0 1241]]


[I 2025-06-28 19:24:56,760] Trial 0 finished with value: 0.5637066668716401 and parameters: {'learning_rate': 0.0002470472304136043, 'num_train_epochs': 5, 'per_device_train_batch_size': 32, 'weight_decay': 0.02}. Best is trial 0 with value: 0.5637066668716401.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


eval/accuracy,▁▁▁▁▁
eval/balanced_accuracy,▁▁▁▁▁
eval/cohen_kappa,▁▁▁▁▁
eval/f1_score,▁▁▁▁▁
eval/hamming_loss,▁▁▁▁▁
eval/jaccard,▁▁▁▁▁
eval/loss,▅▃█▇▁
eval/mcc,▁▁▁▁▁
eval/precision,▁▁▁▁▁
eval/recall,▁▁▁▁▁
eval/runtime,▅▂█▁▁


...

eval/accuracy,▇▇▇▇██▅▃▁
eval/balanced_accuracy,▁▁▁▁▅▇███
eval/cohen_kappa,▁▁▁▁▅████
eval/f1_score,▁▁▁▁▅███▇
eval/hamming_loss,▂▂▂▂▁▁▄▆█
eval/jaccard,▁▁▁▁▅▇███
eval/loss,▁▁▁▁▁▂▄▇█
eval/mcc,▁▁▁▂▇█▇▇▇
eval/precision,▁▁▁▆██▇▇▇
eval/recall,▇▇▇▇██▅▃▁
eval/runtime,▄▂▂▂▁▂▆▅█


Epoch,Training Loss,Validation Loss,F1 Score,Accuracy,Precision,Recall,Mcc,Balanced Accuracy,Cohen Kappa,Jaccard,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.609400,0.620321,0.563707,0.690211,0.476392,0.690211,0.000000,0.500000,0.000000,0.345106,0.309789,1.053300,1706.948000,54.113000
2,0.628600,0.619403,0.563707,0.690211,0.476392,0.690211,0.000000,0.500000,0.000000,0.345106,0.309789,1.053400,1706.883000,54.111000


Confusion Matrix: [[   0  557]
 [   0 1241]]
Confusion Matrix: [[   0  557]
 [   0 1241]]


[I 2025-06-28 19:47:59,025] Trial 19 pruned. 


Best run details:
BestRun(run_id='16', objective=0.6589607956769215, hyperparameters={'learning_rate': 3.714346152977037e-05, 'num_train_epochs': 10, 'per_device_train_batch_size': 32, 'weight_decay': 0.03}, run_summary=None)
Best Hyperparameters Found:
  learning_rate: 3.714346152977037e-05
  num_train_epochs: 10
  per_device_train_batch_size: 32
  weight_decay: 0.03


In [11]:
# %%
#
# Cell 11: Final Model Retraining
#
# Re-training the model with the best hyperparameters
best_hps = best_run.hyperparameters

# Re-initialize the base model
final_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "female", 1: "male"},
    label2id={"female": 0, "male": 1},
    cache_dir=model_cache_path,
    device_map="auto"
)

final_model.config.pad_token_id = tokenizer.pad_token_id

# Define TrainingArguments for the final training run
final_model_output_dir = "../scratch/final_distilbert_degendered_model"
final_training_args = TrainingArguments(
    output_dir=final_model_output_dir,
    per_device_train_batch_size=best_hps["per_device_train_batch_size"],
    per_device_eval_batch_size=best_hps["per_device_train_batch_size"],
    num_train_epochs=best_hps["num_train_epochs"],
    learning_rate=best_hps["learning_rate"],
    weight_decay=best_hps["weight_decay"],
    fp16=True,
    save_strategy="epoch",
    logging_steps=50,
    report_to="wandb",
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="f1_score",
    eval_strategy="epoch",
    save_total_limit=1,
    run_name="final_distilbert_model_training"
)

# Initialize the Trainer for final training
final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Train the final model
final_trainer.train()

# Evaluate the final (best) model
eval_results = final_trainer.evaluate()
print("Final Model Evaluation Results:")
print(eval_results)

# Save the final trained model and tokenizer
final_trainer.save_model(final_model_output_dir)
tokenizer.save_pretrained(final_model_output_dir)

print(f"Final model and tokenizer saved to: {final_model_output_dir}")


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_2387882/3922020647.py:42: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  final_trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1 Score,Accuracy,Precision,Recall,Mcc,Balanced Accuracy,Cohen Kappa,Jaccard,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.621600,0.617729,0.563707,0.690211,0.476392,0.690211,0.000000,0.500000,0.000000,0.345106,0.309789,1.053900,1706.001000,54.083000
2,0.609800,0.606119,0.587946,0.696329,0.687523,0.696329,0.105531,0.515317,0.041355,0.365650,0.303671,1.061500,1693.771000,53.696000
3,0.560400,0.617819,0.617341,0.687430,0.636171,0.687430,0.109802,0.532124,0.080799,0.392153,0.312570,1.057600,1700.092000,53.896000
4,0.465700,0.699960,0.642918,0.679644,0.640809,0.679644,0.145574,0.557159,0.132398,0.419880,0.320356,1.055900,1702.870000,53.984000
5,0.360500,0.836527,0.637344,0.657953,0.629050,0.657953,0.128382,0.556784,0.124333,0.416289,0.342047,1.055200,1703.924000,54.018000
6,0.262600,0.950933,0.636304,0.639043,0.633879,0.639043,0.143750,0.570792,0.143645,0.420341,0.360957,1.054400,1705.311000,54.062000
7,0.193400,1.155373,0.635497,0.632369,0.639127,0.632369,0.155954,0.579316,0.155780,0.422396,0.367631,1.055800,1702.996000,53.988000
8,0.127800,1.324842,0.641864,0.654616,0.634695,0.654616,0.143778,0.566735,0.141821,0.422980,0.345384,1.057200,1700.776000,53.918000
9,0.091900,1.461002,0.643192,0.657953,0.635714,0.657953,0.145501,0.566679,0.142903,0.423894,0.342047,1.058800,1698.187000,53.836000
10,0.064000,1.503724,0.636959,0.648498,0.629923,0.648498,0.133115,0.562304,0.131641,0.418012,0.351502,1.056600,1701.761000,53.949000


Confusion Matrix: [[   0  557]
 [   0 1241]]
Confusion Matrix: [[  22  535]
 [  11 1230]]
Confusion Matrix: [[  69  488]
 [  74 1167]]
Confusion Matrix: [[ 131  426]
 [ 150 1091]]
Confusion Matrix: [[ 162  395]
 [ 220 1021]]
Confusion Matrix: [[218 339]
 [310 931]]
Confusion Matrix: [[245 312]
 [349 892]]
Confusion Matrix: [[187 370]
 [251 990]]
Confusion Matrix: [[ 182  375]
 [ 240 1001]]
Confusion Matrix: [[187 370]
 [262 979]]


Confusion Matrix: [[ 182  375]
 [ 240 1001]]
Final Model Evaluation Results:
{'eval_loss': 1.4610021114349365, 'eval_f1_score': 0.6431922506997386, 'eval_accuracy': 0.6579532814238043, 'eval_precision': 0.635714223155601, 'eval_recall': 0.6579532814238043, 'eval_mcc': 0.14550140790257343, 'eval_balanced_accuracy': 0.566679011684849, 'eval_cohen_kappa': 0.1429029852712974, 'eval_jaccard': 0.4238935146651428, 'eval_hamming_loss': 0.3420467185761958, 'eval_runtime': 1.0655, 'eval_samples_per_second': 1687.528, 'eval_steps_per_second': 53.498, 'epoch': 10.0}
Final model and tokenizer saved to: ../scratch/final_distilbert_degendered_model
